# WiDS Wildfire Survival — Structural-Gated Kernel Stack


In [1]:
# Generates /kaggle/working/submission.csv. Uses only train.csv, test.csv, and sample_submission.csv.
import os, math, warnings, random
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ.setdefault('NUMEXPR_NUM_THREADS','1')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge, HuberRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import pairwise_distances

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    HAS_CAT = False
except Exception:
    HAS_CAT = False

RUN_MODE = 'submit'
OUTPUT_PATH = '/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv'
SEEDS_FULL = (17, 43, 91, 137, 211, 431, 733, 2027)
SEEDS_FAST = (17, 137, 733)
SEEDS = (17, 137, 733)
RANDOM_SEARCH_N = 1200

ID = 'event_id'
TARGET_TIME = 'time_to_hit_hours'
TARGET_EVENT = 'event'
HORIZONS3 = [12, 24, 48]
P_COLS = ['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']

def find_data_dir():
    need = {'train.csv','test.csv','sample_submission.csv'}
    direct = [
        '/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/widsworldwide-globaldathon26',
        '/kaggle/input/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/widsworldwide_globaldathon26',
        '/kaggle/input', '.', '/mnt/data',
        '/mnt/data/contest/anonymous-contest-2-main'
    ]
    for d in direct:
        if os.path.isdir(d) and need.issubset(set(os.listdir(d))):
            return d
    for root in ['/kaggle/input','/mnt/data','.']:
        if not os.path.isdir(root):
            continue
        for path, _, files in os.walk(root):
            if need.issubset(set(files)):
                return path
    raise FileNotFoundError('train.csv, test.csv, sample_submission.csv not found')

DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

time_values = train[TARGET_TIME].to_numpy(float)
event_values = train[TARGET_EVENT].to_numpy(int)

def enforce_monotone(arr):
    arr = np.clip(np.asarray(arr, dtype=float).copy(), 0.0, 1.0)
    for j in range(1, arr.shape[1]):
        arr[:, j] = np.maximum(arr[:, j], arr[:, j-1])
    return arr

def c_index(time, event, risk):
    conc = 0.0
    comp = 0.0
    n = len(time)
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1.0
            if risk[i] > risk[j]:
                conc += 1.0
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def brier_at(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)[valid]
    p = np.clip(prob[valid], 0.0, 1.0)
    return float(np.mean((p - y) ** 2))

def hybrid_score(pred, mode='234'):
    pred = enforce_monotone(pred)
    if mode == '234':
        risk = 0.3 * pred[:, 1] + 0.4 * pred[:, 2] + 0.3 * pred[:, 3]
    elif mode == 'avg4':
        risk = 0.25 * pred[:, 0] + 0.25 * pred[:, 1] + 0.25 * pred[:, 2] + 0.25 * pred[:, 3]
    elif mode == 'early':
        risk = 0.50 * pred[:, 0] + 0.30 * pred[:, 1] + 0.17 * pred[:, 2] + 0.03 * pred[:, 3]
    else:
        risk = pred[:, :3].sum(axis=1) + 0.05 * pred[:, 3]
    ci = c_index(time_values, event_values, risk)
    b24 = brier_at(time_values, event_values, pred[:, 1], 24)
    b48 = brier_at(time_values, event_values, pred[:, 2], 48)
    b72 = brier_at(time_values, event_values, pred[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * ci + 0.7 * (1.0 - wb), ci, wb, b24, b48, b72

def make_features(df):
    r = df.copy()
    dist = np.maximum(r['dist_min_ci_0_5h'].to_numpy(float), 1.0)
    area = np.maximum(r['area_first_ha'].to_numpy(float), 0.0)
    radius_m = np.sqrt(area * 10000.0 / np.pi)
    close = np.maximum(r['closing_speed_m_per_h'].to_numpy(float), 0.0)
    radial = np.maximum(r['radial_growth_rate_m_per_h'].to_numpy(float), 0.0)
    eff_speed = close + radial
    perim = r['num_perimeters_0_5h'].to_numpy(float)
    low = r['low_temporal_resolution_0_5h'].to_numpy(float)
    align = r['alignment_abs'].to_numpy(float)
    hr = r['event_start_hour'].to_numpy(float)
    mo = r['event_start_month'].to_numpy(float)
    dow = r['event_start_dayofweek'].to_numpy(float)

    r['dist_km'] = dist / 1000.0
    r['log_dist'] = np.log1p(dist)
    r['inv_dist_km'] = 1.0 / (dist / 1000.0 + 0.10)
    r['dist_to_5k_m'] = dist - 5000.0
    r['inside5k_m'] = np.maximum(5000.0 - dist, 0.0)
    r['outside5k_m'] = np.maximum(dist - 5000.0, 0.0)
    r['under5k'] = (dist < 5000.0).astype(float)
    r['fire_radius_m'] = radius_m
    r['fire_radius_km'] = radius_m / 1000.0
    r['radius_to_dist'] = radius_m / dist
    r['radius_to_gap'] = radius_m / (np.abs(dist - 5000.0) + 100.0)
    r['eff_closing_speed'] = eff_speed
    r['log_eff_speed'] = np.log1p(eff_speed)
    r['eta_center_h'] = dist / (eff_speed + 1e-3)
    r['eta_gap_h'] = np.maximum(dist - 5000.0, 0.0) / (eff_speed + 1e-3)
    r['log_eta_center'] = np.log1p(np.clip(r['eta_center_h'], 0, 1e6))
    r['log_eta_gap'] = np.log1p(np.clip(r['eta_gap_h'], 0, 1e6))

    r['log_area'] = np.log1p(area)
    r['sqrt_area'] = np.sqrt(area)
    r['area_scaled'] = area / 1000.0
    r['small_area'] = (area < 30.0).astype(float)
    r['tiny_area'] = (area < 10.0).astype(float)
    r['medium_area'] = ((area >= 30.0) & (area < 100.0)).astype(float)
    r['large_area'] = (area > 500.0).astype(float)
    r['giant_area'] = (area > 1500.0).astype(float)

    r['has_motion'] = (perim > 1.0).astype(float)
    r['multi_perim'] = (perim >= 2.0).astype(float)
    r['perim3'] = (perim >= 3.0).astype(float)
    r['many_perim'] = (perim >= 5.0).astype(float)
    r['very_many_perim'] = (perim >= 10.0).astype(float)
    r['perim_log'] = np.log1p(perim)
    r['has_growth'] = (eff_speed > 1e-6).astype(float)

    r['hour_sin'] = np.sin(2.0 * np.pi * hr / 24.0)
    r['hour_cos'] = np.cos(2.0 * np.pi * hr / 24.0)
    r['month_sin'] = np.sin(2.0 * np.pi * mo / 12.0)
    r['month_cos'] = np.cos(2.0 * np.pi * mo / 12.0)
    r['dow_sin'] = np.sin(2.0 * np.pi * dow / 7.0)
    r['dow_cos'] = np.cos(2.0 * np.pi * dow / 7.0)
    r['night_or_late'] = ((hr <= 5.0) | (hr >= 22.0)).astype(float)
    r['evening'] = ((hr >= 18.0) & (hr <= 22.0)).astype(float)
    r['afternoon'] = ((hr >= 14.0) & (hr <= 18.0)).astype(float)
    r['midday'] = ((hr >= 10.0) & (hr <= 17.0)).astype(float)

    r['lowres_small'] = low * r['small_area']
    r['lowres_tiny'] = low * r['tiny_area']
    r['lowres_large'] = low * r['large_area']
    r['lowres_oneperim'] = low * (perim <= 1.0).astype(float)
    r['lowres_giant'] = low * r['giant_area']
    r['tiny_july_late'] = ((low == 1.0) & (area < 10.0) & (hr >= 15.0) & (hr <= 19.0) & ((mo == 7.0) | (mo == 4.0))).astype(float)
    r['small_aug_evening'] = ((low == 1.0) & (area < 30.0) & (hr >= 18.0) & (hr <= 22.0) & (mo == 8.0)).astype(float)
    r['low_med_night'] = ((low == 1.0) & (area >= 30.0) & (area < 300.0) & (hr <= 5.0)).astype(float)
    r['multi2_small_morning'] = ((perim == 2.0) & (low == 0.0) & (area < 30.0) & (hr <= 10.0)).astype(float)

    r['align_motion'] = align * r['has_motion']
    r['align_inv_dist'] = align * r['inv_dist_km']
    r['perim_area'] = r['perim_log'] * r['log_area']
    r['motion_area'] = r['has_motion'] * r['log_area']
    r['growth_area'] = r['log_eff_speed'] * r['log_area']
    r['area_dist_ratio'] = r['log_area'] * r['inv_dist_km']
    r['late_slow_signature'] = r['lowres_oneperim'] * (0.8 * r['small_area'] + 0.5 * r['tiny_area'] + 0.25 * r['midday'])

    r['physics_fast_score'] = (
        (5000.0 - dist) / 1000.0
        + 0.30 * r['log_area']
        + 0.70 * r['multi_perim']
        + 0.65 * r['perim3']
        + 0.75 * r['many_perim']
        + 0.35 * align
        + 0.35 * r['has_growth']
        - 0.85 * low
        - 0.55 * r['small_area']
        - 0.42 * r['tiny_area']
        - 0.25 * r['night_or_late']
        - 0.30 * r['tiny_july_late']
    )
    return r.replace([np.inf, -np.inf], np.nan).fillna(0.0)

train_fe = make_features(train)
test_fe = make_features(test)
drop_cols = [ID, TARGET_TIME, TARGET_EVENT]
feature_cols = [c for c in train_fe.columns if c not in drop_cols]
X_all = train_fe[feature_cols]
X_test = test_fe[feature_cols]
positive_idx = np.where(train['dist_min_ci_0_5h'].to_numpy(float) < 5000.0)[0]
pos_time = time_values[positive_idx]
X_pos = X_all.iloc[positive_idx].reset_index(drop=True)
pos_df = train.iloc[positive_idx].copy().reset_index(drop=True)

hit_train_mask = train['dist_min_ci_0_5h'].to_numpy(float) < 5000.0
hit_test_mask = test['dist_min_ci_0_5h'].to_numpy(float) < 5000.0

def zero_full_from_cond(pos_oof, test_cond, name):
    tr = np.zeros((len(train), 4), dtype=float)
    te = np.zeros((len(test), 4), dtype=float)
    tr[positive_idx, :3] = np.clip(pos_oof, 0.0, 1.0)
    te[hit_test_mask, :3] = np.clip(test_cond[hit_test_mask], 0.0, 1.0)
    tr[:, 3] = 1.0
    te[:, 3] = 1.0
    tr[~hit_train_mask, :3] = 0.0
    te[~hit_test_mask, :3] = 0.0
    return name, enforce_monotone(tr), enforce_monotone(te)

def physics_conditional(df):
    dist = df['dist_min_ci_0_5h'].to_numpy(float)
    area = np.maximum(df['area_first_ha'].to_numpy(float), 0.0)
    perim = df['num_perimeters_0_5h'].to_numpy(float)
    low = df['low_temporal_resolution_0_5h'].to_numpy(float)
    align = df['alignment_abs'].to_numpy(float)
    hr = df['event_start_hour'].to_numpy(float)
    mo = df['event_start_month'].to_numpy(float)
    growth = np.maximum(df['radial_growth_rate_m_per_h'].to_numpy(float), 0.0) + np.maximum(df['closing_speed_m_per_h'].to_numpy(float), 0.0)
    score = np.zeros(len(df), dtype=float)
    score += 1.10 * (perim >= 2.0)
    score += 0.90 * (perim >= 3.0)
    score += 0.90 * (perim >= 5.0)
    score += 0.30 * np.log1p(area)
    score += 0.55 * (align > 0.50)
    score += 0.35 * (align > 0.85)
    score += 0.45 * (growth > 1.0)
    score += 0.26 * (dist < 1200.0)
    score += 0.10 * (dist < 700.0)
    score -= 0.12 * (dist > 3200.0)
    score -= 0.18 * (dist > 4500.0)
    score -= 1.05 * low
    score -= 0.58 * (area < 30.0)
    score -= 0.42 * (area < 10.0)
    score -= 0.12 * ((hr <= 5.0) | (hr >= 23.0))
    score += 0.18 * ((hr >= 18.0) & (hr <= 22.0))
    score -= 0.35 * ((low == 1.0) & (area < 10.0) & (hr >= 15.0) & (hr <= 19.0) & ((mo == 7.0) | (mo == 4.0)))
    score -= 0.12 * ((perim == 2.0) & (area < 2.0) & (hr <= 10.0))
    p12 = expit(score - 0.30)
    p24 = expit(score + 0.95)
    p48 = expit(score + 1.76)
    return enforce_monotone(np.column_stack([p12, p24, p48]))

def group_names(df):
    area = df['area_first_ha'].to_numpy(float)
    perim = df['num_perimeters_0_5h'].to_numpy(float)
    low = df['low_temporal_resolution_0_5h'].to_numpy(float)
    hr = df['event_start_hour'].to_numpy(float)
    mo = df['event_start_month'].to_numpy(float)
    out = []
    for a, p, l, h, m in zip(area, perim, low, hr, mo):
        if p >= 5 and l == 0:
            out.append('multi5')
        elif p >= 3 and l == 0:
            out.append('multi3')
        elif p >= 2 and l == 0 and a < 10:
            out.append('multi2_tiny')
        elif p >= 2 and l == 0 and a < 30:
            out.append('multi2_small')
        elif p >= 2 and l == 0:
            out.append('multi2_other')
        elif l == 1 and a < 10 and 15 <= h <= 19 and m in (4, 7):
            out.append('low_tiny_late_sig')
        elif l == 1 and a < 10:
            out.append('low_tiny')
        elif l == 1 and a < 30:
            out.append('low_small')
        elif l == 1 and a < 100:
            out.append('low_med')
        elif l == 1 and a < 300:
            out.append('low_100_300')
        else:
            out.append('low_big')
    return np.asarray(out)

def group_candidate():
    gp = group_names(pos_df)
    gt = group_names(test)
    global_rates = np.array([(pos_time <= h).mean() for h in HORIZONS3], dtype=float)
    alpha = 4.5
    rates = {}
    for g in np.unique(gp):
        m = gp == g
        n = float(m.sum())
        counts = np.array([np.sum(pos_time[m] <= h) for h in HORIZONS3], dtype=float)
        rates[g] = (counts + alpha * global_rates) / (n + alpha)
    oof = np.zeros((len(pos_time), 3), dtype=float)
    for i, g in enumerate(gp):
        m = gp == g
        m[i] = False
        n = float(m.sum())
        if n <= 0:
            oof[i] = global_rates
        else:
            counts = np.array([np.sum(pos_time[m] <= h) for h in HORIZONS3], dtype=float)
            oof[i] = (counts + alpha * global_rates) / (n + alpha)
    test_cond = np.vstack([rates.get(g, global_rates) for g in gt])
    return zero_full_from_cond(oof, test_cond, 'empirical_group')

def pos_cv_splits():
    bins = np.digitize(pos_time, [12.0, 24.0, 48.0])
    counts = np.bincount(bins)
    n_splits = max(2, min(3, int(counts[counts > 0].min())))
    for seed in SEEDS:
        yield StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X_pos, bins)

def build_logit(seed=17, C=0.25, balanced=False):
    return Pipeline([
        ('scale', RobustScaler()),
        ('lr', LogisticRegression(C=C, solver='liblinear', class_weight='balanced' if balanced else None, random_state=seed, max_iter=2000)),
    ])

def build_knn(seed=17, k=9):
    return Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=k, weights='distance'))])

def build_et_cls(seed=17):
    return ExtraTreesClassifier(n_estimators=120 if RUN_MODE == 'full' else 50, max_depth=3, min_samples_leaf=4, class_weight='balanced_subsample', random_state=seed, n_jobs=1)

def build_gb_cls(seed=17):
    return GradientBoostingClassifier(n_estimators=80 if RUN_MODE == 'full' else 35, learning_rate=0.045, max_depth=1, min_samples_leaf=5, subsample=0.85, random_state=seed)

def build_ridge(seed=17):
    return Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=8.0))])

def build_bayes(seed=17):
    return Pipeline([('scale', StandardScaler()), ('br', BayesianRidge())])

def build_huber(seed=17):
    return Pipeline([('scale', StandardScaler()), ('huber', HuberRegressor(alpha=0.7, epsilon=1.25, max_iter=1000))])

def build_et_reg(seed=17):
    return ExtraTreesRegressor(n_estimators=130 if RUN_MODE == 'full' else 55, max_depth=3, min_samples_leaf=4, random_state=seed, n_jobs=1)

def positive_classifier_candidate(builder, name):
    y_mat = np.column_stack([(pos_time <= h).astype(int) for h in HORIZONS3])
    oof = np.zeros((len(pos_time), 3), dtype=float)
    cnt = np.zeros(len(pos_time), dtype=float)
    test_acc = np.zeros((len(test), 3), dtype=float)
    for split_iter in pos_cv_splits():
        for tr_idx, va_idx in split_iter:
            cnt[va_idx] += 1.0
            for j, h in enumerate(HORIZONS3):
                y = y_mat[:, j]
                if len(np.unique(y[tr_idx])) < 2:
                    oof[va_idx, j] += y[tr_idx].mean()
                else:
                    model = builder(seed=17 + j)
                    model.fit(X_pos.iloc[tr_idx], y[tr_idx])
                    oof[va_idx, j] += model.predict_proba(X_pos.iloc[va_idx])[:, 1]
    oof /= cnt[:, None]
    for seed in SEEDS:
        for j, h in enumerate(HORIZONS3):
            y = y_mat[:, j]
            if len(np.unique(y)) < 2:
                test_acc[:, j] += y.mean()
            else:
                model = builder(seed=seed + 31 * j)
                model.fit(X_pos, y)
                test_acc[:, j] += model.predict_proba(X_test)[:, 1]
    test_cond = test_acc / len(SEEDS)
    return zero_full_from_cond(oof, test_cond, name)

def positive_regression_candidate(builder, name):
    y_log = np.log1p(pos_time)
    oof = np.zeros((len(pos_time), 3), dtype=float)
    cnt = np.zeros(len(pos_time), dtype=float)
    test_acc = np.zeros((len(test), 3), dtype=float)
    for split_iter in pos_cv_splits():
        for tr_idx, va_idx in split_iter:
            model = builder(seed=17)
            model.fit(X_pos.iloc[tr_idx], y_log[tr_idx])
            mu_va = model.predict(X_pos.iloc[va_idx])
            mu_tr = model.predict(X_pos.iloc[tr_idx])
            resid = y_log[tr_idx] - mu_tr
            for j, h in enumerate(HORIZONS3):
                z = np.log1p(h) - mu_va
                oof[va_idx, j] += np.mean(resid[None, :] <= z[:, None], axis=1)
            cnt[va_idx] += 1.0
    oof /= cnt[:, None]
    for seed in SEEDS:
        model = builder(seed=seed)
        model.fit(X_pos, y_log)
        mu_test = model.predict(X_test)
        mu_train = model.predict(X_pos)
        resid = y_log - mu_train
        for j, h in enumerate(HORIZONS3):
            z = np.log1p(h) - mu_test
            test_acc[:, j] += np.mean(resid[None, :] <= z[:, None], axis=1)
    test_cond = test_acc / len(SEEDS)
    return zero_full_from_cond(oof, test_cond, name)

def kernel_feature_frame(df):
    f = pd.DataFrame(index=df.index)
    dist = np.maximum(df['dist_min_ci_0_5h'].to_numpy(float), 1.0)
    area = np.maximum(df['area_first_ha'].to_numpy(float), 0.0)
    perim = df['num_perimeters_0_5h'].to_numpy(float)
    low = df['low_temporal_resolution_0_5h'].to_numpy(float)
    hr = df['event_start_hour'].to_numpy(float)
    mo = df['event_start_month'].to_numpy(float)
    dow = df['event_start_dayofweek'].to_numpy(float)
    align = df['alignment_abs'].to_numpy(float)
    growth = np.maximum(df['radial_growth_rate_m_per_h'].to_numpy(float), 0.0) + np.maximum(df['closing_speed_m_per_h'].to_numpy(float), 0.0)
    f['log_area'] = np.log1p(area)
    f['sqrt_area'] = np.sqrt(area)
    f['log_dist'] = np.log1p(dist)
    f['dist_km'] = dist / 1000.0
    f['inv_dist'] = 1.0 / (dist / 1000.0 + 0.10)
    f['perim_log'] = np.log1p(perim)
    f['perim'] = perim
    f['low'] = low
    f['align'] = align
    f['growth_log'] = np.log1p(growth)
    f['dt'] = df['dt_first_last_0_5h'].to_numpy(float)
    for period, arr, nm in [(24, hr, 'hour'), (12, mo, 'month'), (7, dow, 'dow')]:
        f[nm + '_sin'] = np.sin(2.0 * np.pi * arr / period)
        f[nm + '_cos'] = np.cos(2.0 * np.pi * arr / period)
    f['evening'] = ((hr >= 18) & (hr <= 22)).astype(float)
    f['night'] = ((hr <= 5) | (hr >= 23)).astype(float)
    f['afternoon'] = ((hr >= 14) & (hr <= 18)).astype(float)
    f['tiny'] = (area < 10).astype(float)
    f['small'] = (area < 30).astype(float)
    f['medium'] = ((area >= 30) & (area < 100)).astype(float)
    f['big'] = (area >= 300).astype(float)
    f['huge'] = (area >= 1000).astype(float)
    f['multi'] = (perim >= 2).astype(float)
    f['perim3'] = (perim >= 3).astype(float)
    f['perim5'] = (perim >= 5).astype(float)
    f['perim10'] = (perim >= 10).astype(float)
    f['low_tiny'] = f['low'] * f['tiny']
    f['low_small'] = f['low'] * f['small']
    f['low_big'] = f['low'] * f['big']
    f['tiny_late_sig'] = ((low == 1) & (area < 10) & (hr >= 15) & (hr <= 19) & ((mo == 7) | (mo == 4))).astype(float)
    f['score'] = 1.3 * f['multi'] + 0.8 * f['perim3'] + 0.6 * f['perim5'] + 0.3 * np.log1p(area) + 0.3 * align - 0.9 * low - 0.55 * f['small'] - 0.3 * f['tiny'] + 0.1 * f['evening'] - 0.25 * f['tiny_late_sig']
    return f.replace([np.inf, -np.inf], 0.0).fillna(0.0)

K_pos = kernel_feature_frame(pos_df)
K_test = kernel_feature_frame(test)
K_SETS = {
    'all': list(K_pos.columns),
    'core': ['log_area','sqrt_area','log_dist','dist_km','inv_dist','perim_log','low','align','hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos','evening','night','tiny','small','big','huge','score'],
    'lowtime': ['log_area','sqrt_area','log_dist','dist_km','low','hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos','evening','night','afternoon','tiny','small','medium','big','huge','tiny_late_sig'],
}

def kernel_survival_candidate(fs='all', bw=1.7, k=21, alpha=0.5, name=None):
    cols = K_SETS[fs]
    X = K_pos[cols].to_numpy(float)
    Xt = K_test[cols].to_numpy(float)
    scaler = RobustScaler().fit(X)
    Z = scaler.transform(X)
    Zt = scaler.transform(Xt)
    D = pairwise_distances(Z, Z)
    Dt = pairwise_distances(Zt, Z)
    global_rates = np.array([(pos_time <= h).mean() for h in HORIZONS3], dtype=float)
    def one(d, exclude=None):
        dd = d.copy()
        if exclude is not None:
            dd[exclude] = np.inf
        idx = np.where(np.isfinite(dd))[0]
        if k is not None and k < len(idx):
            idx = idx[np.argpartition(dd[idx], k)[:k]]
        w = np.exp(-0.5 * (dd[idx] / bw) ** 2)
        if w.sum() < 1e-12:
            w = np.ones(len(idx), dtype=float)
        return np.array([(np.sum(w * (pos_time[idx] <= h)) + alpha * global_rates[j]) / (w.sum() + alpha) for j, h in enumerate(HORIZONS3)], dtype=float)
    oof = np.vstack([one(D[i], i) for i in range(len(pos_time))])
    test_cond = np.vstack([one(Dt[i], None) for i in range(len(test))])
    if name is None:
        name = f'kernel_{fs}_bw{bw}_k{k}_a{alpha}'
    return zero_full_from_cond(oof, test_cond, name)

candidate_names = []
candidate_train = []
candidate_test = []

def add_candidate(tup):
    nm, tr, te = tup
    candidate_names.append(nm)
    candidate_train.append(enforce_monotone(tr))
    candidate_test.append(enforce_monotone(te))

add_candidate(zero_full_from_cond(physics_conditional(pos_df), physics_conditional(test), 'physics_timing'))
add_candidate(group_candidate())

for fs, bw, k, alpha in [
    ('all', 1.7, 21, 0.5), ('all', 3.0, 7, 0.5), ('all', 3.0, 5, 0.5),
    ('all', 2.2, 21, 0.5), ('all', 1.7, None, 0.5), ('core', 1.7, None, 2.0),
    ('core', 1.3, None, 2.0), ('lowtime', 1.3, 9, 2.0), ('lowtime', 2.2, 13, 0.5),
]:
    try:
        add_candidate(kernel_survival_candidate(fs=fs, bw=bw, k=k, alpha=alpha))
    except Exception as e:
        print('skip kernel', fs, bw, k, str(e)[:100])

for nm, builder in [
    ('pos_logit', lambda seed=17: build_logit(seed, C=0.16, balanced=False)),
    ('pos_logit_balanced', lambda seed=17: build_logit(seed, C=0.35, balanced=True)),
    ('pos_knn', lambda seed=17: build_knn(seed, k=9)),
    ('pos_extratrees_cls', build_et_cls),
    ('pos_gb_stump', build_gb_cls),
]:
    try:
        add_candidate(positive_classifier_candidate(builder, nm))
    except Exception as e:
        print('skip', nm, str(e)[:100])

for nm, builder in [
    ('pos_ridge_residual_cdf', build_ridge),
    ('pos_bayes_residual_cdf', build_bayes),
    ('pos_huber_residual_cdf', build_huber),
    ('pos_extratrees_residual_cdf', build_et_reg),
]:
    try:
        add_candidate(positive_regression_candidate(builder, nm))
    except Exception as e:
        print('skip', nm, str(e)[:100])

if HAS_LGB:
    def build_lgb_cls(seed=17):
        return lgb.LGBMClassifier(objective='binary', n_estimators=95 if RUN_MODE == 'full' else 45, learning_rate=0.035, num_leaves=4, max_depth=2, min_child_samples=5, subsample=0.85, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=4.0, random_state=seed, n_jobs=1, verbosity=-1)
    def build_lgb_reg(seed=17):
        return lgb.LGBMRegressor(objective='regression', n_estimators=95 if RUN_MODE == 'full' else 45, learning_rate=0.035, num_leaves=4, max_depth=2, min_child_samples=5, subsample=0.85, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=4.0, random_state=seed, n_jobs=1, verbosity=-1)
    try:
        add_candidate(positive_classifier_candidate(build_lgb_cls, 'pos_lgb_cls'))
    except Exception as e:
        print('skip pos_lgb_cls', str(e)[:100])
    try:
        add_candidate(positive_regression_candidate(build_lgb_reg, 'pos_lgb_residual_cdf'))
    except Exception as e:
        print('skip pos_lgb_residual_cdf', str(e)[:100])

if HAS_XGB:
    def build_xgb_cls(seed=17):
        return xgb.XGBClassifier(objective='binary:logistic', n_estimators=85 if RUN_MODE == 'full' else 40, learning_rate=0.035, max_depth=2, min_child_weight=4.0, subsample=0.85, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0, random_state=seed, n_jobs=1, eval_metric='logloss')
    def build_xgb_reg(seed=17):
        return xgb.XGBRegressor(objective='reg:squarederror', n_estimators=85 if RUN_MODE == 'full' else 40, learning_rate=0.035, max_depth=2, min_child_weight=4.0, subsample=0.85, colsample_bytree=0.75, reg_alpha=1.0, reg_lambda=5.0, random_state=seed, n_jobs=1)
    try:
        add_candidate(positive_classifier_candidate(build_xgb_cls, 'pos_xgb_cls'))
    except Exception as e:
        print('skip pos_xgb_cls', str(e)[:100])
    try:
        add_candidate(positive_regression_candidate(build_xgb_reg, 'pos_xgb_residual_cdf'))
    except Exception as e:
        print('skip pos_xgb_residual_cdf', str(e)[:100])

if HAS_CAT:
    def build_cat_cls(seed=17):
        return CatBoostClassifier(iterations=120 if RUN_MODE == 'full' else 55, depth=2, learning_rate=0.035, l2_leaf_reg=12.0, loss_function='Logloss', random_seed=seed, verbose=False, allow_writing_files=False, thread_count=1)
    def build_cat_reg(seed=17):
        return CatBoostRegressor(iterations=120 if RUN_MODE == 'full' else 55, depth=2, learning_rate=0.035, l2_leaf_reg=12.0, loss_function='RMSE', random_seed=seed, verbose=False, allow_writing_files=False, thread_count=1)
    try:
        add_candidate(positive_classifier_candidate(build_cat_cls, 'pos_cat_cls'))
    except Exception as e:
        print('skip pos_cat_cls', str(e)[:100])
    try:
        add_candidate(positive_regression_candidate(build_cat_reg, 'pos_cat_residual_cdf'))
    except Exception as e:
        print('skip pos_cat_residual_cdf', str(e)[:100])

candidate_train = np.stack(candidate_train, axis=0)
candidate_test = np.stack(candidate_test, axis=0)

scores234 = np.array([hybrid_score(candidate_train[i], '234')[0] for i in range(len(candidate_names))])
scores_avg = np.array([hybrid_score(candidate_train[i], 'avg4')[0] for i in range(len(candidate_names))])
scores_early = np.array([hybrid_score(candidate_train[i], 'early')[0] for i in range(len(candidate_names))])
blend_scores = 0.74 * scores234 + 0.18 * scores_avg + 0.08 * scores_early
for i, nm in enumerate(candidate_names):
    sc = hybrid_score(candidate_train[i], '234')
    print(f'{i:02d} {nm:32s} score234={sc[0]:.6f} ci={sc[1]:.6f} wb={sc[2]:.6f} avg4={scores_avg[i]:.6f}')

def blend_arrays(stack, w):
    return enforce_monotone(np.tensordot(w, stack, axes=(0, 0)))

def optimize_weights(scores, objective_mode='234', seed=20260424):
    n = len(scores)
    temperature = 0.0038
    prior = np.exp((scores - scores.max()) / temperature)
    prior = prior / prior.sum()
    rng = np.random.default_rng(seed)
    best_w = prior.copy()
    best_sc = hybrid_score(blend_arrays(candidate_train, best_w), objective_mode)[0]
    top_order = np.argsort(-scores)
    proposals = [prior, np.ones(n) / n]
    for k in [2, 3, 5, min(8, n), min(12, n)]:
        w = np.zeros(n)
        w[top_order[:k]] = 1.0 / k
        proposals.append(w)
    for w in proposals:
        sc = hybrid_score(blend_arrays(candidate_train, w), objective_mode)[0]
        if sc > best_sc:
            best_sc = sc
            best_w = w.copy()
    alpha_center = 1.0 + 35.0 * prior
    for _ in range(RANDOM_SEARCH_N):
        if rng.random() < 0.72:
            w = rng.dirichlet(alpha_center)
        else:
            k = int(rng.integers(2, min(7, n) + 1))
            chosen = rng.choice(top_order[:min(n, 12)], size=k, replace=False)
            w = np.zeros(n)
            w[chosen] = rng.dirichlet(np.ones(k) * 1.4)
        sc = hybrid_score(blend_arrays(candidate_train, w), objective_mode)[0]
        if sc > best_sc:
            best_sc = sc
            best_w = w.copy()
    final_w = 0.68 * best_w + 0.32 * prior
    return final_w / final_w.sum(), best_sc

w234, _ = optimize_weights(blend_scores, '234', 20260424)
wavg, _ = optimize_weights(scores_avg, 'avg4', 20260425)
wearly, _ = optimize_weights(scores_early, 'early', 20260426)
final_w = 0.82 * w234 + 0.10 * wavg + 0.08 * wearly
final_w = final_w / final_w.sum()
blend_train = blend_arrays(candidate_train, final_w)
blend_test = blend_arrays(candidate_test, final_w)
print('blend_precal_234', hybrid_score(blend_train, '234'))
print('blend_precal_avg4', hybrid_score(blend_train, 'avg4'))

calibrated_train = blend_train.copy()
calibrated_test = blend_test.copy()
for j, h, alpha in [(0, 12, 0.16), (1, 24, 0.26), (2, 48, 0.28)]:
    valid = ~((event_values == 0) & (time_values < h))
    y = ((event_values == 1) & (time_values <= h)).astype(float)
    if len(np.unique(y[valid])) >= 2:
        ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
        ir.fit(blend_train[valid, j], y[valid])
        calibrated_train[:, j] = (1.0 - alpha) * blend_train[:, j] + alpha * ir.predict(blend_train[:, j])
        calibrated_test[:, j] = (1.0 - alpha) * blend_test[:, j] + alpha * ir.predict(blend_test[:, j])

calibrated_train[~hit_train_mask, :3] = 0.0
calibrated_test[~hit_test_mask, :3] = 0.0
calibrated_train[:, 3] = 1.0
calibrated_test[:, 3] = 1.0
calibrated_train = enforce_monotone(calibrated_train)
calibrated_test = enforce_monotone(calibrated_test)
print('blend_cal_234', hybrid_score(calibrated_train, '234'))
print('blend_cal_avg4', hybrid_score(calibrated_train, 'avg4'))

# Mild train-derived late-pattern adjustment; applied only if it improves OOF composite.
def structural_adjust(arr, df, tiny_scale=1.0, multi_boost=0.0, small_morning_scale=1.0):
    out = arr.copy()
    area = df['area_first_ha'].to_numpy(float)
    perim = df['num_perimeters_0_5h'].to_numpy(float)
    low = df['low_temporal_resolution_0_5h'].to_numpy(float)
    hr = df['event_start_hour'].to_numpy(float)
    mo = df['event_start_month'].to_numpy(float)
    tiny_late = (low == 1) & (area < 10) & (hr >= 15) & (hr <= 19) & ((mo == 7) | (mo == 4))
    out[tiny_late, 0] *= tiny_scale
    out[tiny_late, 1] *= tiny_scale
    out[tiny_late, 2] *= 0.85 + 0.15 * tiny_scale
    sm_morning = (perim == 2) & (low == 0) & (area < 2.0) & (hr <= 10)
    out[sm_morning, 0] *= small_morning_scale
    out[sm_morning, 1] *= 0.90 + 0.10 * small_morning_scale
    multi_strong = (perim >= 5) & (low == 0)
    out[multi_strong, :3] = out[multi_strong, :3] * (1.0 - multi_boost) + multi_boost * np.array([0.94, 0.985, 0.993])
    out[:, 3] = 1.0
    return enforce_monotone(out)

best_train = calibrated_train.copy()
best_test = calibrated_test.copy()
def composite_local(p):
    return 0.74 * hybrid_score(p, '234')[0] + 0.18 * hybrid_score(p, 'avg4')[0] + 0.08 * hybrid_score(p, 'early')[0]
best_score = composite_local(best_train)
for tiny_scale in [1.0, 0.92, 0.84, 0.76, 0.68]:
    for multi_boost in [0.0, 0.02, 0.04, 0.06]:
        for small_morning_scale in [1.0, 0.90, 0.80]:
            tr_try = structural_adjust(calibrated_train, train, tiny_scale, multi_boost, small_morning_scale)
            tr_try[~hit_train_mask, :3] = 0.0
            tr_try[:, 3] = 1.0
            sc = composite_local(tr_try)
            if sc > best_score:
                best_score = sc
                best_train = tr_try
                best_test = structural_adjust(calibrated_test, test, tiny_scale, multi_boost, small_morning_scale)
                best_test[~hit_test_mask, :3] = 0.0
                best_test[:, 3] = 1.0

calibrated_train = enforce_monotone(best_train)
calibrated_test = enforce_monotone(best_test)
calibrated_train[~hit_train_mask, :3] = 0.0
calibrated_test[~hit_test_mask, :3] = 0.0
calibrated_train[:, 3] = 1.0
calibrated_test[:, 3] = 1.0
print('blend_final_234', hybrid_score(calibrated_train, '234'))
print('blend_final_avg4', hybrid_score(calibrated_train, 'avg4'))

print('Final blend weights:')
for nm, w, sc in sorted(zip(candidate_names, final_w, blend_scores), key=lambda z: -z[1]):
    if w > 1e-4:
        print(f'{w:8.5f} {sc:.6f} {nm}')

submission = sample[[ID]].copy()
pred_df = pd.DataFrame({ID: test[ID].values})
pred_df[P_COLS] = calibrated_test
submission = submission.merge(pred_df, on=ID, how='left')
submission[P_COLS] = submission[P_COLS].astype(float)
submission[P_COLS] = enforce_monotone(submission[P_COLS].to_numpy(float))
submission['prob_72h'] = 1.0
submission = submission[[ID] + P_COLS]

assert len(submission) == len(sample)
assert submission[ID].is_unique
assert set(submission[ID]) == set(sample[ID])
assert np.isfinite(submission[P_COLS].to_numpy()).all()
assert ((submission[P_COLS] >= 0.0) & (submission[P_COLS] <= 1.0)).all().all()
assert (submission[P_COLS].diff(axis=1).iloc[:, 1:] >= -1e-12).all().all()
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True) if os.path.dirname(OUTPUT_PATH) else None
submission.to_csv(OUTPUT_PATH, index=False)
print('saved', OUTPUT_PATH, submission.shape)
print(submission.describe())


00 physics_timing                   score234=0.975289 ci=0.946754 wb=0.012481 avg4=0.975289
01 empirical_group                  score234=0.972905 ci=0.935906 wb=0.011239 avg4=0.973091
02 kernel_all_bw1.7_k21_a0.5        score234=0.972010 ci=0.936154 wb=0.012623 avg4=0.974693
03 kernel_all_bw3.0_k7_a0.5         score234=0.966465 ci=0.918350 wb=0.012915 avg4=0.974489
04 kernel_all_bw3.0_k5_a0.5         score234=0.964678 ci=0.910980 wb=0.012308 avg4=0.974019
05 kernel_all_bw2.2_k21_a0.5        score234=0.972480 ci=0.939384 wb=0.013337 avg4=0.974045
06 kernel_all_bw1.7_kNone_a0.5      score234=0.972258 ci=0.937148 wb=0.012695 avg4=0.974121
07 kernel_core_bw1.7_kNone_a2.0     score234=0.970986 ci=0.935989 wb=0.014016 avg4=0.972625
08 kernel_core_bw1.3_kNone_a2.0     score234=0.971556 ci=0.936568 wb=0.013449 avg4=0.973071
09 kernel_lowtime_bw1.3_k9_a2.0     score234=0.967486 ci=0.921663 wb=0.012876 avg4=0.970169
10 kernel_lowtime_bw2.2_k13_a0.5    score234=0.967924 ci=0.924395 wb=0.013421 av